In [ ]:
import pandas as pd

# Load the medical dataset directly from a secure URL
url = "https://storage.googleapis.com/download.tensorflow.org/data/heart.csv"
df = pd.read_csv(url)

# Let's take a peek at the first 5 patients
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0,fixed,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3,normal,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2,reversible,0
3,37,1,3,130,250,0,0,187,0,3.5,3,0,normal,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0,normal,0


In [ ]:
print(f"Dataset Shape: {df.shape}")
print("-" * 40)

print("Missing values per column:")
print(df.isnull().sum())

Dataset Shape: (303, 14)
----------------------------------------
Missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('target', axis=1)
y = df.target

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
categorical_cols =['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)
print('Prepocessing Pipeline Built Successfully!')

In [ ]:
from xgboost import XGBClassifier
model = XGBClassifier(n_estimators=100, learning_rate=0.05, random_state=42, eval_metric='logloss')

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

pipeline.fit(X_train, y_train)
print('XGBoost pipeline trained successfully!')

XGBoost pipeline trained successfully!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

prediction = pipeline.predict(X_valid)

accuracy = accuracy_score(y_valid, prediction)
print(f'Validation Accuracy = {accuracy * 100:.2f}%')
print('-' * 50)

classification = classification_report(y_valid, prediction)
print("Classification Report=\n", classification)

Validation Accuracy = 83.61%
--------------------------------------------------
Classification Report=
               precision    recall  f1-score   support

           0       0.87      0.91      0.89        44
           1       0.73      0.65      0.69        17

    accuracy                           0.84        61
   macro avg       0.80      0.78      0.79        61
weighted avg       0.83      0.84      0.83        61

